# FEVER Data Preparation v3 — Lightweight Retrieval + NLI-Filtered NEI

This version preserves FEVER gold evidence extraction and adds a **small local retrieval + pretrained NLI filtering pipeline** for generating more realistic NEI examples. It does not use the FACTLENS retrieval system.

## NEI generation approach
1. Build a bounded sentence corpus from Wikipedia using reservoir sampling.
2. Retrieve a few claim-relevant sentences with a lightweight TF-IDF index (scikit-learn; sparse matrix).
3. Score each retrieved sentence with a small pretrained NLI model (`cross-encoder/nli-MiniLM2-L6-H768`).
4. Keep a sentence for an NEI example only when the NLI model predicts **neutral** with sufficient confidence and low entailment/contradiction probabilities.
5. Drop NEI claims for which no neutral candidate is found; do not silently relabel supporting/refuting evidence as NEI.

This is still an **automatic weak-labeling pipeline**: a pretrained NLI model can make mistakes. Audit the generated examples and keep the original FEVER paper-test split untouched for final reporting.

## 0. Configuration

In [1]:
import json
import unicodedata
import random
import hashlib
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np

SEED = 42

DATA_DIR = Path("fever-data/raw")
PROCESSED_DATA_DIR = Path("fever-data/processed")
WIKI_DIR = DATA_DIR / "wiki-pages" / "wiki-pages"

TRAIN_FILE = DATA_DIR / "train.jsonl"
DEV_FILE = DATA_DIR / "shared_task_dev.jsonl"
PAPER_DEV_FILE = DATA_DIR / "paper_dev.jsonl"
PAPER_TEST_FILE = DATA_DIR / "paper_test.jsonl"

# Limit alternative gold evidence groups per positive claim.
MAX_GROUPS_PER_CLAIM = 3

# Lightweight retrieval corpus. Increase only if RAM permits.
NEI_SENTENCE_POOL_SIZE = 100_000
RETRIEVAL_TOP_K = 8
MAX_NEI_EVIDENCE_SENTENCES = 3

# NLI acceptance thresholds. A candidate must be neutral and have low
# entailment/contradiction probabilities to be retained as NEI evidence.
NEUTRAL_MIN_PROB = 0.60
ENTAILMENT_MAX_PROB = 0.20
CONTRADICTION_MAX_PROB = 0.20

# Small NLI model; runs on GPU if available, otherwise CPU.
NLI_MODEL_NAME = "cross-encoder/nli-MiniLM2-L6-H768"
NLI_BATCH_SIZE = 32
NLI_MAX_LENGTH = 256

N_AUDIT_SAMPLES = 8
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data directory:", DATA_DIR.resolve())
print("Wikipedia directory:", WIKI_DIR.resolve())
print("Processed output directory:", PROCESSED_DATA_DIR.resolve())

Raw data directory: /run/media/shinigami/Lakshay/Coding/Git/Factlens/experiments/classifier/fever-data/raw
Wikipedia directory: /run/media/shinigami/Lakshay/Coding/Git/Factlens/experiments/classifier/fever-data/raw/wiki-pages/wiki-pages
Processed output directory: /run/media/shinigami/Lakshay/Coding/Git/Factlens/experiments/classifier/fever-data/processed


## 1. Load FEVER JSONL files

In [2]:
def load_jsonl(file_path):
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path.resolve()}")
    rows = []
    with file_path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {file_path}, line {line_no}") from exc
    return rows

train_raw = load_jsonl(TRAIN_FILE)
dev_raw = load_jsonl(DEV_FILE)
paper_dev_raw = load_jsonl(PAPER_DEV_FILE)
paper_test_raw = load_jsonl(PAPER_TEST_FILE)

for name, data in [
    ("train", train_raw), ("dev", dev_raw),
    ("paper_dev", paper_dev_raw), ("paper_test", paper_test_raw)
]:
    print(f"{name:10s}: {len(data):,} claims")

train     : 145,449 claims
dev       : 19,998 claims
paper_dev : 9,999 claims
paper_test: 9,999 claims


## 2. Collect and extract gold Wikipedia evidence

In [3]:
def normalize_page_id(page_id):
    if page_id is None:
        return None
    return unicodedata.normalize("NFC", str(page_id))

def get_required_pages(datasets):
    pages = set()
    for data in datasets:
        for item in data:
            if item.get("label") == "NOT ENOUGH INFO":
                continue
            for group in item.get("evidence") or []:
                for annotation in group:
                    if len(annotation) >= 4:
                        page_id = normalize_page_id(annotation[2])
                        if page_id is not None:
                            pages.add(page_id)
    return pages

required_pages = get_required_pages([
    train_raw, dev_raw, paper_dev_raw, paper_test_raw
])
print(f"Unique Wikipedia pages needed for annotated evidence: {len(required_pages):,}")

Unique Wikipedia pages needed for annotated evidence: 14,533


In [4]:
def build_wiki_lookup(wiki_dir, required_pages):
    wiki_lookup = {}
    wiki_files = sorted(Path(wiki_dir).glob("*.jsonl"))
    if not wiki_files:
        raise FileNotFoundError(f"No Wikipedia .jsonl files found under {Path(wiki_dir).resolve()}")

    print(f"Scanning {len(wiki_files):,} Wikipedia files...")
    for file_path in wiki_files:
        with file_path.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    page = json.loads(line)
                except json.JSONDecodeError:
                    continue

                page_id = normalize_page_id(page.get("id"))
                if page_id not in required_pages:
                    continue

                sentence_lookup = {}
                for row in page.get("lines", "").split("\n"):
                    if not row.strip():
                        continue
                    parts = row.split("\t", 2)
                    if len(parts) < 2:
                        continue
                    try:
                        sentence_id = int(parts[0].strip())
                    except ValueError:
                        continue
                    sentence_text = parts[1].strip()
                    if sentence_text:
                        sentence_lookup[sentence_id] = sentence_text

                wiki_lookup[page_id] = sentence_lookup

    missing_pages = required_pages - set(wiki_lookup)
    print(f"Extracted pages: {len(wiki_lookup):,}")
    print(f"Missing pages: {len(missing_pages):,}")
    return wiki_lookup

wiki_lookup = build_wiki_lookup(WIKI_DIR, required_pages)

Scanning 109 Wikipedia files...
Extracted pages: 14,533
Missing pages: 0


## 3. Convert each complete gold evidence group into a row

In [5]:
def preprocess_fever(data, wiki_lookup, split_name):
    processed = []
    incomplete_groups = []
    no_group_claims = []

    for item in data:
        label = item.get("label")
        claim_id = item.get("id")
        claim = item.get("claim", "")

        if label == "NOT ENOUGH INFO":
            processed.append({
                "id": claim_id,
                "claim": claim,
                "label": "NEI",
                "evidence_group_id": None,
                "evidence": [],
                "split": split_name
            })
            continue

        if label not in ("SUPPORTS", "REFUTES"):
            continue

        groups = item.get("evidence") or []
        if not groups:
            no_group_claims.append(claim_id)
            continue

        for group_idx, group in enumerate(groups):
            evidence_sentences = []
            missing = []

            for annotation in group:
                if len(annotation) < 4:
                    missing.append(("malformed_annotation", annotation))
                    continue

                page_id = normalize_page_id(annotation[2])
                sentence_id = annotation[3]
                sentence = wiki_lookup.get(page_id, {}).get(sentence_id)

                if sentence is None:
                    missing.append((page_id, sentence_id))
                else:
                    evidence_sentences.append({
                        "page_id": page_id,
                        "sentence_id": sentence_id,
                        "text": sentence
                    })

            # Keep only complete groups; never train on a partial gold group.
            if missing or not evidence_sentences:
                incomplete_groups.append({
                    "claim_id": claim_id,
                    "group_index": group_idx,
                    "missing": missing
                })
                continue

            processed.append({
                "id": claim_id,
                "claim": claim,
                "label": label,
                "evidence_group_id": group_idx,
                "evidence": evidence_sentences,
                "split": split_name
            })

    print(
        f"{split_name}: {len(processed):,} rows; "
        f"{len(incomplete_groups):,} incomplete groups skipped; "
        f"{len(no_group_claims):,} positive claims with no groups"
    )
    return processed, incomplete_groups, no_group_claims

train_processed, train_incomplete, train_no_group = preprocess_fever(train_raw, wiki_lookup, "train")
dev_processed, dev_incomplete, dev_no_group = preprocess_fever(dev_raw, wiki_lookup, "dev")
paper_dev_processed, paper_dev_incomplete, paper_dev_no_group = preprocess_fever(paper_dev_raw, wiki_lookup, "paper_dev")
paper_test_processed, paper_test_incomplete, paper_test_no_group = preprocess_fever(paper_test_raw, wiki_lookup, "paper_test")

train: 257,115 rows; 0 incomplete groups skipped; 0 positive claims with no groups
dev: 31,391 rows; 0 incomplete groups skipped; 0 positive claims with no groups
paper_dev: 15,775 rows; 0 incomplete groups skipped; 0 positive claims with no groups
paper_test: 15,616 rows; 0 incomplete groups skipped; 0 positive claims with no groups


## 4. Build a bounded Wikipedia sentence corpus

We use reservoir sampling to keep a reproducible subset of Wikipedia sentences in memory. This is the document collection for our lightweight local retrieval step. It is not the FACTLENS retrieval system.

In [7]:

def build_sentence_pool(wiki_dir, pool_size=100_000, seed=42):
    rng = random.Random(seed)
    reservoir = []
    total_seen = 0

    wiki_files = sorted(Path(wiki_dir).glob("*.jsonl"))

    if not wiki_files:
        raise FileNotFoundError(
            f"No Wikipedia JSONL files found in: {Path(wiki_dir).resolve()}"
        )

    print(f"Scanning {len(wiki_files):,} Wikipedia files...")

    for file_path in wiki_files:
        with file_path.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                try:
                    page = json.loads(line)
                except json.JSONDecodeError:
                    continue

                page_id = normalize_page_id(page.get("id"))
                raw_lines = page.get("lines", "")

                if not raw_lines:
                    continue

                # IMPORTANT: Actual newline and tab delimiters
                for row in raw_lines.splitlines():
                    if not row.strip():
                        continue

                    parts = row.split("\t", 2)

                    if len(parts) < 2:
                        continue

                    try:
                        sentence_id = int(parts[0].strip())
                    except ValueError:
                        continue

                    sentence_text = parts[1].strip()

                    if not sentence_text:
                        continue

                    total_seen += 1

                    record = {
                        "page_id": page_id,
                        "sentence_id": sentence_id,
                        "text": sentence_text
                    }

                    # Reservoir sampling
                    if len(reservoir) < pool_size:
                        reservoir.append(record)
                    else:
                        j = rng.randrange(total_seen)

                        if j < pool_size:
                            reservoir[j] = record

    print(f"Corpus sentences scanned: {total_seen:,}")
    print(f"Random pool size: {len(reservoir):,}")

    if not reservoir:
        raise ValueError(
            "No Wikipedia sentences found. Check WIKI_DIR and JSONL format."
        )

    return reservoir


sentence_pool = build_sentence_pool(
    WIKI_DIR,
    pool_size=NEI_SENTENCE_POOL_SIZE,
    seed=SEED
)


Scanning 109 Wikipedia files...
Corpus sentences scanned: 25,247,890
Random pool size: 100,000


## 5. Limit alternative evidence groups per claim

A claim with many alternative gold groups should not automatically contribute many times more rows than a claim with one group. This cell deterministically samples at most `MAX_GROUPS_PER_CLAIM` complete groups per positive claim.

The cap is applied independently within each split. It does not move claims between splits.

In [8]:
def stable_int(value):
    digest = hashlib.sha256(str(value).encode("utf-8")).hexdigest()[:16]
    return int(digest, 16)

def cap_positive_groups(rows, max_groups=3, seed=42):
    by_claim = defaultdict(list)
    nei_rows = []
    for row in rows:
        if row["label"] in ("SUPPORTS", "REFUTES"):
            by_claim[row["id"]].append(row)
        else:
            nei_rows.append(row)

    kept = []
    for claim_id, claim_rows in by_claim.items():
        claim_rows = sorted(
            claim_rows,
            key=lambda r: (r["evidence_group_id"] is None, r["evidence_group_id"])
        )
        if max_groups is not None and len(claim_rows) > max_groups:
            rng = random.Random(seed + stable_int(claim_id))
            claim_rows = rng.sample(claim_rows, max_groups)
        kept.extend(claim_rows)

    kept.extend(nei_rows)
    random.Random(seed).shuffle(kept)
    return kept

train_capped = cap_positive_groups(train_processed, MAX_GROUPS_PER_CLAIM, SEED)
dev_capped = cap_positive_groups(dev_processed, MAX_GROUPS_PER_CLAIM, SEED + 1)
paper_dev_capped = cap_positive_groups(paper_dev_processed, MAX_GROUPS_PER_CLAIM, SEED + 2)
paper_test_capped = cap_positive_groups(paper_test_processed, MAX_GROUPS_PER_CLAIM, SEED + 3)

for name, before, after in [
    ("train", train_processed, train_capped),
    ("dev", dev_processed, dev_capped),
    ("paper_dev", paper_dev_processed, paper_dev_capped),
    ("paper_test", paper_test_processed, paper_test_capped)
]:
    print(f"{name:10s}: {len(before):,} -> {len(after):,} rows")

train     : 257,115 -> 194,125 rows
dev       : 31,391 -> 25,623 rows
paper_dev : 15,775 -> 12,822 rows
paper_test: 15,616 -> 12,801 rows


## 6. Lightweight retrieval + NLI verification for NEI examples

**Retrieval:** TF-IDF cosine similarity over the bounded Wikipedia sentence pool. This is sparse and relatively lightweight.

**Verification:** a small pretrained NLI model scores each retrieved sentence against the claim. We retain only candidates with a strong `neutral` score and low `entailment` and `contradiction` scores. If no candidate passes, that NEI claim is skipped.

The NLI model is a filtering aid, not a human ground-truth annotator. Review the audit samples and thresholds before using the generated data.

In [9]:
# Build a sparse TF-IDF index over the sampled Wikipedia sentences.
# Install scikit-learn only if it is not already available in the runtime.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import linear_kernel
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

corpus_texts = [s["text"] for s in sentence_pool]

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=100_000,
    min_df=2,
    dtype=np.float32,
    sublinear_tf=True
)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_texts)
tfidf_matrix = normalize(tfidf_matrix, norm="l2", copy=False)

print("TF-IDF matrix:", tfidf_matrix.shape, "non-zero:", tfidf_matrix.nnz)

# Load a compact 3-way NLI model.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
nli_model.to(device)
nli_model.eval()

print("NLI device:", device)
print("NLI labels:", nli_model.config.id2label)

def normalize_nli_label(label):
    label = str(label).lower()
    if "neutral" in label:
        return "neutral"
    if "entail" in label:
        return "entailment"
    if "contrad" in label:
        return "contradiction"
    return label

id_to_nli_label = {
    int(idx): normalize_nli_label(label)
    for idx, label in nli_model.config.id2label.items()
}
required_nli_labels = {"neutral", "entailment", "contradiction"}
if not required_nli_labels.issubset(set(id_to_nli_label.values())):
    raise ValueError(
        f"Unexpected NLI label mapping: {id_to_nli_label}. "
        "Check the selected model before generating NEI rows."
    )

def retrieve_candidates(claim, top_k=8):
    query_vec = tfidf_vectorizer.transform([claim])
    if query_vec.nnz == 0:
        return []
    scores = linear_kernel(query_vec, tfidf_matrix).ravel()
    k = min(top_k, len(scores))
    if k == 0:
        return []
    top_idx = np.argpartition(scores, -k)[-k:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    return [
        (int(idx), float(scores[idx]))
        for idx in top_idx
        if scores[idx] > 0
    ]

@torch.inference_mode()
def score_nli_pairs(pairs, batch_size=32):
    # Each pair is (premise=evidence, hypothesis=claim).
    all_probs = []
    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]
        premises = [p[0] for p in batch]
        hypotheses = [p[1] for p in batch]
        encoded = nli_tokenizer(
            premises,
            hypotheses,
            padding=True,
            truncation=True,
            max_length=NLI_MAX_LENGTH,
            return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        logits = nli_model(**encoded).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)
    return all_probs

def get_prob(prob_vector, wanted_label):
    for class_idx, label in id_to_nli_label.items():
        if label == wanted_label:
            return float(prob_vector[class_idx])
    return 0.0

def retrieve_and_verify_nei(claim, top_k=RETRIEVAL_TOP_K,
                            max_evidence=MAX_NEI_EVIDENCE_SENTENCES):
    candidates = retrieve_candidates(claim, top_k=top_k)
    if not candidates:
        return []

    pairs = [(sentence_pool[idx]["text"], claim) for idx, _ in candidates]
    probabilities = score_nli_pairs(pairs, batch_size=NLI_BATCH_SIZE)

    accepted = []
    for (idx, retrieval_score), probs in zip(candidates, probabilities):
        neutral_p = get_prob(probs, "neutral")
        entail_p = get_prob(probs, "entailment")
        contradict_p = get_prob(probs, "contradiction")

        if (
            neutral_p >= NEUTRAL_MIN_PROB
            and entail_p <= ENTAILMENT_MAX_PROB
            and contradict_p <= CONTRADICTION_MAX_PROB
        ):
            sentence = sentence_pool[idx]
            accepted.append({
                "page_id": sentence["page_id"],
                "sentence_id": sentence["sentence_id"],
                "text": sentence["text"],
                "retrieval_score": retrieval_score,
                "nli_neutral_prob": neutral_p,
                "nli_entailment_prob": entail_p,
                "nli_contradiction_prob": contradict_p
            })
        if len(accepted) >= max_evidence:
            break

    return accepted

def build_classifier_dataset(processed_rows, seed=42):
    rng = random.Random(seed)
    final_rows = []
    skipped_nei = 0

    for row in processed_rows:
        if row["label"] in ("SUPPORTS", "REFUTES"):
            if not row["evidence"]:
                continue
            final_rows.append({
                "id": row["id"],
                "claim": row["claim"],
                "label": row["label"],
                "evidence_group_id": row["evidence_group_id"],
                "evidence": row["evidence"],
                "evidence_source": "gold"
            })

        elif row["label"] == "NEI":
            verified_evidence = retrieve_and_verify_nei(row["claim"])
            if not verified_evidence:
                skipped_nei += 1
                continue

            # Strip internal scores from the classifier input while retaining
            # the sentence/page identifiers and text.
            evidence = [
                {k: item[k] for k in ("page_id", "sentence_id", "text")}
                for item in verified_evidence
            ]
            final_rows.append({
                "id": row["id"],
                "claim": row["claim"],
                "label": "NEI",
                "evidence_group_id": None,
                "evidence": evidence,
                "evidence_source": "sampled_corpus"
            })

    rng.shuffle(final_rows)
    return final_rows, skipped_nei

print("NLI retrieval and verification functions are ready.")

/home/shinigami/miniconda3/envs/factlens/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TF-IDF matrix: (100000, 100000) non-zero: 1232810


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10304.45it/s]

NLI device: cpu
NLI labels: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}
NLI retrieval and verification functions are ready.


In [10]:
train_classifier_data, train_nei_skipped = build_classifier_dataset(train_capped, SEED)
dev_classifier_data, dev_nei_skipped = build_classifier_dataset(dev_capped, SEED + 1)
paper_dev_classifier_data, paper_dev_nei_skipped = build_classifier_dataset(paper_dev_capped, SEED + 2)
paper_test_classifier_data, paper_test_nei_skipped = build_classifier_dataset(paper_test_capped, SEED + 3)

print("Generated datasets:")
for name, data, skipped in [
    ("train", train_classifier_data, train_nei_skipped),
    ("dev", dev_classifier_data, dev_nei_skipped),
    ("paper_dev", paper_dev_classifier_data, paper_dev_nei_skipped),
    ("paper_test", paper_test_classifier_data, paper_test_nei_skipped),
]:
    print(f"{name:10s}: {len(data):,} rows; skipped NEI claims without neutral evidence: {skipped:,}")

Generated datasets:
train     : 186,217 rows; skipped NEI claims without neutral evidence: 7,908
dev       : 24,242 rows; skipped NEI claims without neutral evidence: 1,381
paper_dev : 12,086 rows; skipped NEI claims without neutral evidence: 736
paper_test: 12,156 rows; skipped NEI claims without neutral evidence: 645


## 7. Dataset integrity checks and statistics

In [11]:
def inspect_classifier_dataset(data, name):
    labels = Counter(row["label"] for row in data)
    sources = Counter(row["evidence_source"] for row in data)
    sizes = Counter(len(row["evidence"]) for row in data)
    ids = [row["id"] for row in data]

    print(f"\n{name}")
    print("-" * 55)
    print(f"Rows: {len(data):,}")
    print("Labels:", dict(labels))
    print("Evidence sources:", dict(sources))
    print("Evidence sentence counts:", dict(sorted(sizes.items())))
    print(f"Unique claim IDs: {len(set(ids)):,} / {len(ids):,} rows")
    print(f"Rows with empty evidence: {sum(not r['evidence'] for r in data):,}")

    assert all(r["label"] in {"SUPPORTS", "REFUTES", "NEI"} for r in data)
    assert all(len(r["evidence"]) > 0 for r in data)
    assert all(r["evidence_source"] == "gold" for r in data if r["label"] in {"SUPPORTS", "REFUTES"})
    assert all(r["evidence_source"] == "sampled_corpus" for r in data if r["label"] == "NEI")

for name, dataset in [
    ("TRAIN", train_classifier_data),
    ("DEV", dev_classifier_data),
    ("PAPER DEV", paper_dev_classifier_data),
    ("PAPER TEST", paper_test_classifier_data)
]:
    inspect_classifier_dataset(dataset, name)


TRAIN
-------------------------------------------------------
Rows: 186,217
Labels: {'SUPPORTS': 115573, 'REFUTES': 42913, 'NEI': 27731}
Evidence sources: {'gold': 158486, 'sampled_corpus': 27731}
Evidence sentence counts: {1: 142298, 2: 25942, 3: 16853, 4: 545, 5: 234, 6: 125, 7: 91, 8: 35, 9: 25, 10: 22, 11: 9, 12: 6, 13: 6, 14: 7, 15: 2, 16: 5, 19: 2, 22: 2, 23: 2, 25: 1, 29: 1, 30: 1, 32: 1, 33: 1, 41: 1}
Unique claim IDs: 137,541 / 186,217 rows
Rows with empty evidence: 0

DEV
-------------------------------------------------------
Rows: 24,242
Labels: {'NEI': 5285, 'SUPPORTS': 9508, 'REFUTES': 9449}
Evidence sources: {'sampled_corpus': 5285, 'gold': 18957}
Evidence sentence counts: {1: 17968, 2: 3015, 3: 3137, 4: 54, 5: 25, 6: 14, 7: 9, 8: 10, 9: 1, 10: 3, 11: 1, 12: 2, 17: 1, 20: 1, 25: 1}
Unique claim IDs: 18,617 / 24,242 rows
Rows with empty evidence: 0

PAPER DEV
-------------------------------------------------------
Rows: 12,086
Labels: {'REFUTES': 4738, 'NEI': 2597, 'SUPP

## 8. Claim-ID split leakage check

In [16]:
def claim_id_set(raw_data):
    return {row["id"] for row in raw_data}

split_ids = {
    "train": claim_id_set(train_raw),
    "dev": claim_id_set(dev_raw),
    "paper_dev": claim_id_set(paper_dev_raw),
    "paper_test": claim_id_set(paper_test_raw)
}

names = list(split_ids)
b = "train"  # Compare all splits against the training set for leakage detection
for i, a in enumerate(names):
    if a == b:
        continue
    overlap = split_ids[a] & split_ids[b]
    print(f"{a} vs {b}: {len(overlap):,} overlapping claim IDs")
    assert not overlap, f"Claim-ID leakage detected between {a} and {b}!"

dev vs train: 0 overlapping claim IDs
paper_dev vs train: 0 overlapping claim IDs
paper_test vs train: 0 overlapping claim IDs


## 9. Retrieved NEI evidence audit

Inspect retrieved evidence and its NLI scores before training. This audit is important because the NLI model is an automatic filter and can still make mistakes.

In [17]:
def audit_nei_examples(data, name, n=8, seed=42):
    nei = [r for r in data if r["label"] == "NEI"]
    if not nei:
        print(f"{name}: no generated NEI rows")
        return
    rng = random.Random(seed)
    sample = rng.sample(nei, min(n, len(nei)))
    print(f"\\n{name}: retrieved + NLI-filtered NEI audit")
    for row in sample:
        print("\\nClaim:", row["claim"])
        for ev in row["evidence"]:
            print("Evidence:", ev["text"])

audit_nei_examples(train_classifier_data, "TRAIN", N_AUDIT_SAMPLES, SEED)

\nTRAIN: retrieved + NLI-filtered NEI audit
\nClaim: Heartlight is a pop song
Evidence: Almighty Records specialises in pop song remixes , dance-pop , and hi-NRG .
Evidence: Musically constructed as a doo-wop-style pop song , it is regarded as a light-hearted track by critics and includes instruments such as a drum machine , percussions , keyboards and a saxophone .
Evidence: Category : Pop standards
\nClaim: John D. Rockefeller smoked a cigar every day.
Evidence: Its early collection was built up by the American sculptor , art dealer and collector George Grey Barnard , and acquired by John D. Rockefeller , Jr. in 1925 .
\nClaim: Prakash Jha is Indian and German.
Evidence: The film 's music was scored by G. V. Prakash Kumar .
Evidence: The chain was run by Ved Prakash Arya from its founding until March 2004 .
\nClaim: The Wire is set in World War II.
Evidence: In World War II he
Evidence: It was used by the US Army in World War II .
\nClaim: Georgia Institute of Technology's basketball

## 10. Save classifier JSONL files

In [19]:
def save_jsonl(data, file_path):
    with file_path.open("w", encoding="utf-8") as f:
        for row in data:
            # Keep classifier-compatible fields only.
            output_row = {
                "id": row["id"],
                "claim": row["claim"],
                "label": row["label"],
                "evidence_group_id": row["evidence_group_id"],
                "evidence": row["evidence"],
                "evidence_source": row["evidence_source"]
            }
            f.write(json.dumps(output_row, ensure_ascii=False) + "\n")

save_jsonl(train_classifier_data, PROCESSED_DATA_DIR / "train_classifier.jsonl")
save_jsonl(dev_classifier_data, PROCESSED_DATA_DIR / "dev_classifier.jsonl")
save_jsonl(paper_dev_classifier_data, PROCESSED_DATA_DIR / "paper_dev_classifier.jsonl")
save_jsonl(paper_test_classifier_data, PROCESSED_DATA_DIR / "paper_test_classifier.jsonl")

print("Saved:")
for filename in [
    "train_classifier.jsonl", "dev_classifier.jsonl",
    "paper_dev_classifier.jsonl", "paper_test_classifier.jsonl"
]:
    p = PROCESSED_DATA_DIR / filename
    print(f"{p}: {p.stat().st_size / (1024**2):.2f} MB")

Saved:
fever-data/processed/train_classifier.jsonl: 78.59 MB
fever-data/processed/dev_classifier.jsonl: 9.93 MB
fever-data/processed/paper_dev_classifier.jsonl: 4.90 MB
fever-data/processed/paper_test_classifier.jsonl: 5.02 MB


## 11. Final schema and label audit

The classifier notebook should read the four files from `fever-data/processed`.

Expected row schema:
- `id`: original FEVER claim ID
- `claim`: claim text
- `label`: SUPPORTS / REFUTES / NEI
- `evidence_group_id`: integer for positive gold group, null for NEI
- `evidence`: list of `{page_id, sentence_id, text}`
- `evidence_source`: `gold` or `sampled_corpus` (NEI rows are TF-IDF-retrieved and NLI-filtered)

**Important:** This v2 changes the NEI source string from `sampled_corpus` to `sampled_corpus`. If your classifier notebook validates exact strings, update its NEI check to accept `sampled_corpus` (or change the string below back to `sampled_corpus`). The recommended update is to use `sampled_corpus` because it describes the process more accurately.

In [20]:
for split_name, dataset in [
    ("train", train_classifier_data),
    ("dev", dev_classifier_data),
    ("paper_dev", paper_dev_classifier_data),
    ("paper_test", paper_test_classifier_data)
]:
    assert dataset, f"{split_name} is empty"
    for row in dataset[:min(100, len(dataset))]:
        assert row["claim"] and isinstance(row["claim"], str)
        assert row["label"] in {"SUPPORTS", "REFUTES", "NEI"}
        assert isinstance(row["evidence"], list) and len(row["evidence"]) > 0
        for ev in row["evidence"]:
            assert isinstance(ev.get("text"), str) and ev["text"].strip()
            assert "page_id" in ev and "sentence_id" in ev

print("Final schema checks passed.")
print("\nV2 preprocessing complete.")

Final schema checks passed.

V2 preprocessing complete.
